In [ ]:
"""
Created on Tues Mar 29 16:12:01 2022
@author: Oumbeg
"""

import re
import camelot
from bs4 import BeautifulSoup
import os
import pandas as pd
from time import sleep
import datetime
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

now=datetime.datetime.now()
filename= 'ZA SARB data {}.xlsx'.format(str(now).replace(":",".")[:-7])
scriptfolder=os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder')

try:
  os.mkdir(tempfolder)
except:
  prevfiles=os.listdir(tempfolder)
  for prf in prevfiles:
    os.remove(os.path.join(tempfolder, prf))
  print('The directory tempfolder already exists.')

print("Running ZA SARB Web Scraping Tool v.1.0")


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}                                                
   
regdict={'ZA SARB 1': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices', 
#          'ZA SARB 2': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices', 
#          'ZA SARB 3': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices', 
#          'ZA SARB 4': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices', 
#          'ZA SARB 5': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices',
#          'ZA SARB 6': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices',
         'ZA SARB 7': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/insurers-list',
         'ZA SARB 8': 'https://www.resbank.co.za/content/dam/sarb/what-we-do/prudential-regulation/PA%20registered%20co-operative%20banks%20as%20at%20October%202020.pdf',
         'ZA SARB 9': 'https://www.resbank.co.za/content/dam/sarb/what-we-do/prudential-regulation/PA%20registered%20CFI%27s%20as%20at%2031%20December%202021.pdf'}


print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))
chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
    "plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

def findEmail(myData):
    """
    This function finds email in a string.
    :param myData: string
    :return: String
    """
    regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')
    email = regex.findall(myData)
    return ' '.join([str(elem) for elem in email])

def findUrl(mydata):
    regex = re.compile(r"(?i)\b((?:https?://|www\d{2,4,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'\".,<>?«»“”‘’]))")
    url = regex.findall(mydata)
    return ' '.join([str(elem) for elem in [x[0] for x in url]])
    
# try to create an empty folder "tempfolder"
try:
    os.mkdir(tempfolder)
except:
    prevfiles=os.listdir(tempfolder)
    os.chdir(tempfolder)
    for prf in prevfiles:
        os.remove(prf)
    print('The directory tempfolder already exists.')
os.chdir(tempfolder)##only if files are going to be downloaded here

processdate=now.strftime('%Y-%m-%d')
pattern = re.compile('^[0-9]*$')
my_dict = {'Banks in Liquidation': '6', 'Branches of Foreign Banks': '1', 'Foreign Bank Representatives': '2',
          'Foreign Controlled Banks': '3', 'Locally Controlled Banks': '4', 'Mutual Banks':'5'}
cumul_prev_len_as2 = 0

for reg in regdict:
    
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(5)
    
    if int(reg[-1]) == 1:
    
        # the block where to find the links 
        soup = BeautifulSoup(driver.page_source, "html.parser")
        bloc = soup.find("div",{"class":"bankCategories__bank-types"})
        a1_s = bloc.find_all('a')
        #print(len(a1_s))
    
        for i in range(1,len(a1_s)+1):
            the_code = my_dict.get(a1_s[i-1].text.strip())
            print('-------',a1_s[i-1].text.strip(),'-------')
            if the_code != '0':
                element1 = driver.find_element(By.XPATH, '//*[@id="base-page-3fe1ae529b"]/div[3]/div/div[2]/div/div[4]/div[2]/div['+str(i)+']/a')
                driver.execute_script("arguments[0].click();", element1)
                sleep(3)
        
                soup = BeautifulSoup(driver.page_source, "html.parser")
                bloc = soup.find("div",{"class":"bankCategories2__bank-types"})
                a2_s = bloc.find_all('a')
                len_as2 = len(a2_s)-cumul_prev_len_as2
                
                #print('**length a2_s = ',len(a2_s))
                print('***length a2_s_cum = ',len_as2)
    
                for j in range(1,len_as2+1):
                    element2 = driver.find_element(By.XPATH, '//*[@id="base-page-3fe1ae529b"]/div[3]/div/div[2]/div/div[5]/div[2]/div['+str(j)+']/a')
                    driver.execute_script("arguments[0].click();", element2)
                    sleep(3)
        
                    soup = BeautifulSoup(driver.page_source, "html.parser")
                    bloc = soup.find("div",{"class":"bankCategories3__bank-types"})
                    name = bloc.find_all("div",{"class":"trade-name"})
            
                    bank = name[0].text.split('Address:')[0].strip()
                    address = name[0].find_all("div",{"class":"bankCategories3__address"})[0].text.replace('Address:','').strip()
                    code = name[0].find_all("div",{"class":"bankCategories3__postalCode"})[0].text.replace('Postal Code:','').strip()
                    phone = name[0].find_all("div",{"class":"bankCategories3__telephone"})[0].text.replace('H/O Telephone:','').strip()
                    fax = name[0].find_all("div",{"class":"bankCategories3__fax"})[0].text.replace('H/O Fax:','').strip()
                    web = name[0].find_all("div",{"class":"bankCategories3__webAddress"})[0].text.replace('Web Address:','').strip()
                    complete_address = address+', '+code
            
                    print('Name ',j,': ', bank)
                    sqldict['Name'].append(bank)
                    sqldict['Address_1'].append(complete_address)
                    sqldict['Phone'].append(phone)
                    sqldict['Fax'].append(fax)
                    sqldict['Website'].append(web)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListCode'].append(the_code)
                    sqldict['RegCode'].append('SARB')
                    sqldict['RegCtry'].append('ZA')
                    sqldict['RegulationType'].append('Supervised')
            
                    # Fill the rest with empty string
                    for key in sqldict.keys():
                        if len(sqldict['Name']) > len(sqldict[key]):
                            sqldict[key].append('')
        
                    driver.get(regdict[reg])
                    sleep(3)
            
                    element1 = driver.find_element(By.XPATH, '//*[@id="base-page-3fe1ae529b"]/div[3]/div/div[2]/div/div[4]/div[2]/div['+str(i)+']/a')
                    driver.execute_script("arguments[0].click();", element1)
                    sleep(1)
                    
                cumul_prev_len_as2 = len(a2_s)-cumul_prev_len_as2
                
    elif int(reg[-1]) == 7:
    
        # the block where to find the links 
        soup = BeautifulSoup(driver.page_source, "html.parser")
        table = soup.find("table",{"class":"table"})
        trs = table.find_all('tr')
        #print('-->',len(trs))
        for k in range(1,len(trs)):
            #print(trs[k].find_all('td')[0].text.strip())
            sqldict['Name'].append(trs[k].find_all('td')[0].text.strip())
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListCode'].append(reg[-1])
            sqldict['RegCode'].append('SARB')
            sqldict['RegCtry'].append('ZA')
            sqldict['RegulationType'].append('Supervised')
            
            # Fill the rest with empty string
            for key in sqldict.keys():
                if len(sqldict['Name']) > len(sqldict[key]):
                    sqldict[key].append('')
                    
    elif int(reg[-1]) in [8,9]:
        while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
            print('Waiting for file to download')
            sleep(1)
    
        print(os.listdir(tempfolder))
        pdf_file = os.listdir(tempfolder)[0]
        filePath = os.path.join(tempfolder, pdf_file)
            
        tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')
    
        # Iterate over the tables of each pages
        for i in range(tables.n):
            df_Table = tables[i].df
                
            for j in range(0, len(df_Table)):
                if int(reg[-1]) == 8 and df_Table[0][j].strip().split()[0] not in ['Register', 'Name']:
                    print('-->',''.join(df_Table[0][j].strip().splitlines()))
                    
                    sqldict['Name'].append(''.join(df_Table[0][j].strip().splitlines()))
                    sqldict['InternalID_1'].append(df_Table[2][j].strip())
                    sqldict['InternalID_1_type'].append('Registration Number as a Cooperative Bank')
                    sqldict['InternalID_2'].append(df_Table[3][j].strip())
                    sqldict['InternalID_2_type'].append('Registration Number as a Co-operative')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListCode'].append(reg[-1])
                    sqldict['RegCode'].append('SARB')
                    sqldict['RegCtry'].append('ZA')
                    sqldict['RegulationType'].append('Supervised')
                    
                    # Fill the rest with empty string
                    for key in sqldict.keys():
                        if len(sqldict['Name']) > len(sqldict[key]):
                            sqldict[key].append('')
                            
                elif int(reg[-1]) == 9 and df_Table[0][j].strip().split()[0] not in ['Register', 'Name'] and i != tables.n-1:
                    
                    #print('<-> Adress: ', ''.join(df_Table[2][j].strip().splitlines()))
                    #print('|_| Details: ', ''.join(df_Table[4][j].strip().splitlines()).split())
                    tel = ''
                    mail = ''
                    web = ''
                    p1 = ''
                    p2 = ''
                    detail_list = ''.join(df_Table[4][j].strip().splitlines()).split()
                    
                    if len(detail_list[0]) == 1:
                        p1 = detail_list[0]
                        
                    gate = True
                        
                    for item in detail_list:
                        
                        if len(findEmail(item)) > 0 and gate:
                            mail = p1+findEmail(item)
                            gate = False
                        elif len(findUrl(item.replace('ww.','www.').replace('ttp','http'))) > 0 or 'ww.' in item:
                            web = item.replace('ww.','www.').replace('ttp','http')
                        elif len(pattern.findall(item)) > 0:
                            if tel == '':
                                tel = '0'+item
                            else:
                                tel = tel+' '+item
                                
                    #print('- Email: ',mail)
                    #print('- Url: ',web)
                    #print('- Tel: ',tel,'\n')
                    
                    sqldict['Name'].append(''.join(df_Table[0][j].strip().splitlines()))
                    sqldict['Address_1'].append(''.join(df_Table[2][j].strip().splitlines()))
                    sqldict['Phone'].append(tel.strip())
                    sqldict['Email'].append(mail.strip())
                    sqldict['Website'].append(web.strip())
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListCode'].append(reg[-1])
                    sqldict['RegCode'].append('SARB')
                    sqldict['RegCtry'].append('ZA')
                    sqldict['RegulationType'].append('Supervised')
                    
                    # Fill the rest with empty string
                    for key in sqldict.keys():
                        if len(sqldict['Name']) > len(sqldict[key]):
                            sqldict[key].append('')
                            
        os.remove(filePath)
                            
    
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()
    
    
    